# Versao 12 - Pre-Processamento

Este notebook explicita o pipeline de entrada da `versao12`. A ideia e preservar a riqueza estrutural da `versao10`, aproveitar a limpeza de features da `versao11` e preparar os tensores necessarios para a arquitetura profunda e hierarquica nova.

## O que muda em relacao as versoes anteriores

- em relacao a `versao10`: a entrada deixa de ser tratada como sequencia unica e passa a alimentar uma arquitetura hierarquica por janelas;
- em relacao a `versao11`: a limpeza de features vazias e mantida, mas o recorte observacional do treino nao e herdado;
- em relacao a `versao9`: a separacao entre variaveis continuas e de estado agora convive com mascaras operacionais e multitarefa.

Assim, os artefatos produzidos continuam sendo `X_seq`, `X_tab`, `X_missing`, `X_frozen`, `y`, `y_step_class`, `y_step_state` e `source_id`, mas agora voltados para uma rede mais completa.

In [1]:
from pathlib import Path
import importlib
import sys

import numpy as np
import pandas as pd

ROOT = Path.cwd()
PROJECT_ROOT = ROOT.parent if ROOT.name == "versao12" else ROOT
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import versao12.pipeline_v12 as pipeline_v12

pipeline_v12 = importlib.reload(pipeline_v12)
SELECTED_FEATURE_COLUMNS = pipeline_v12.SELECTED_FEATURE_COLUMNS
SELECTED_STATE_SENSOR_COLUMNS = pipeline_v12.SELECTED_STATE_SENSOR_COLUMNS
SELECTED_CONTINUOUS_SENSOR_COLUMNS = pipeline_v12.SELECTED_CONTINUOUS_SENSOR_COLUMNS
ALL_NULL_FEATURE_COLUMNS = pipeline_v12.ALL_NULL_FEATURE_COLUMNS
resolve_feature_group_indices = pipeline_v12.resolve_feature_group_indices
load_bundle = pipeline_v12.load_bundle
load_split_arrays = pipeline_v12.load_split_arrays
prepare_classification_artifacts = pipeline_v12.prepare_classification_artifacts

continuous_indices, state_indices = resolve_feature_group_indices(
    SELECTED_FEATURE_COLUMNS,
    SELECTED_STATE_SENSOR_COLUMNS,
)

print("Features removidas:", ALL_NULL_FEATURE_COLUMNS)
print("Numero de features mantidas:", len(SELECTED_FEATURE_COLUMNS))
print("Numero de features continuas:", len(continuous_indices))
print("Numero de features de estado:", len(state_indices))


Features removidas: ['ABER-CKGL', 'ABER-CKP', 'P-JUS-BS', 'P-JUS-CKP', 'P-MON-CKGL', 'P-MON-SDV-P', 'PT-P', 'QBS', 'T-MON-CKP']
Numero de features mantidas: 18
Numero de features continuas: 9
Numero de features de estado: 9


## Funcao-chave

A `versao12` precisa saber exatamente quais colunas alimentam cada ramo da rede. O bloco abaixo e a funcao usada para separar indices continuos e indices de estado.

In [2]:
def resolve_feature_group_indices(
    feature_columns: list[str],
    state_columns: list[str] | None = None,
) -> tuple[list[int], list[int]]:
    state_names = set(state_columns if state_columns is not None else SELECTED_STATE_SENSOR_COLUMNS)
    state_indices = [
        idx for idx, column_name in enumerate(feature_columns)
        if column_name in state_names
    ]
    continuous_indices = [
        idx for idx, column_name in enumerate(feature_columns)
        if column_name not in state_names
    ]
    if not continuous_indices and not state_indices:
        raise ValueError("E necessario ter ao menos uma feature continua ou de estado.")
    return continuous_indices, state_indices


In [3]:
DATASET_ROOT = PROJECT_ROOT / "3W" / "dataset"
RUN_NAME = "classificacao_v12_profunda_hierarquica_multitarefa"

artifacts = prepare_classification_artifacts(
    dataset_root=DATASET_ROOT,
    run_name=RUN_NAME,
    random_state=42,
    sequence_length=180,
)

bundle = load_bundle(artifacts.bundle_path)
train_arrays = load_split_arrays(artifacts.split_npz_paths["train"])
validation_arrays = load_split_arrays(artifacts.split_npz_paths["validation"])
test_arrays = load_split_arrays(artifacts.split_npz_paths["test"])

print("Run dir:", artifacts.run_dir)
print("Bundle path:", artifacts.bundle_path)
print("Features selecionadas:", bundle.selected_columns)
print("State columns:", bundle.state_columns)
print("Continuous columns:", bundle.continuous_columns)
print("X_tab dimension:", len(bundle.statistical_feature_names))


Run dir: /home/tiagoriosrocha/Desktop/lstm-w3/artifacts/reports_v12/classificacao_v12_profunda_hierarquica_multitarefa
Bundle path: /home/tiagoriosrocha/Desktop/lstm-w3/artifacts/reports_v12/classificacao_v12_profunda_hierarquica_multitarefa/bundle_v12.json
Features selecionadas: ['ESTADO-DHSV', 'ESTADO-M1', 'ESTADO-M2', 'ESTADO-PXO', 'ESTADO-SDV-GL', 'ESTADO-SDV-P', 'ESTADO-W1', 'ESTADO-W2', 'ESTADO-XO', 'P-ANULAR', 'P-JUS-CKGL', 'P-MON-CKP', 'P-PDG', 'P-TPT', 'QGL', 'T-JUS-CKP', 'T-PDG', 'T-TPT']
State columns: ['ESTADO-DHSV', 'ESTADO-M1', 'ESTADO-M2', 'ESTADO-PXO', 'ESTADO-SDV-GL', 'ESTADO-SDV-P', 'ESTADO-W1', 'ESTADO-W2', 'ESTADO-XO']
Continuous columns: ['P-ANULAR', 'P-JUS-CKGL', 'P-MON-CKP', 'P-PDG', 'P-TPT', 'QGL', 'T-JUS-CKP', 'T-PDG', 'T-TPT']
X_tab dimension: 162


In [4]:
resumo_arrays = pd.DataFrame(
    [
        {
            "split": "train",
            "X_seq": train_arrays["X_seq"].shape,
            "X_tab": train_arrays["X_tab"].shape,
            "X_missing": train_arrays["X_missing"].shape,
            "X_frozen": train_arrays["X_frozen"].shape,
            "y": train_arrays["y"].shape,
            "y_step_class": train_arrays["y_step_class"].shape,
            "y_step_state": train_arrays["y_step_state"].shape,
        },
        {
            "split": "validation",
            "X_seq": validation_arrays["X_seq"].shape,
            "X_tab": validation_arrays["X_tab"].shape,
            "X_missing": validation_arrays["X_missing"].shape,
            "X_frozen": validation_arrays["X_frozen"].shape,
            "y": validation_arrays["y"].shape,
            "y_step_class": validation_arrays["y_step_class"].shape,
            "y_step_state": validation_arrays["y_step_state"].shape,
        },
        {
            "split": "test",
            "X_seq": test_arrays["X_seq"].shape,
            "X_tab": test_arrays["X_tab"].shape,
            "X_missing": test_arrays["X_missing"].shape,
            "X_frozen": test_arrays["X_frozen"].shape,
            "y": test_arrays["y"].shape,
            "y_step_class": test_arrays["y_step_class"].shape,
            "y_step_state": test_arrays["y_step_state"].shape,
        },
    ]
)
display(resumo_arrays)


,split,X_seq,X_tab,X_missing,X_frozen,y,y_step_class,y_step_state
0,train,"(1559, 180, 18)","(1559, 162)","(1559, 180, 18)","(1559, 180, 18)","(1559,)","(1559, 180)","(1559, 180)"
1,validation,"(334, 180, 18)","(334, 162)","(334, 180, 18)","(334, 180, 18)","(334,)","(334, 180)","(334, 180)"
2,test,"(335, 180, 18)","(335, 162)","(335, 180, 18)","(335, 180, 18)","(335,)","(335, 180)","(335, 180)"


## Leitura final

O pre-processamento da `versao12` continua fiel a ideia de que o `3W` deve ser lido em varios niveis ao mesmo tempo: dinamica temporal, contexto agregado, irregularidades operacionais e supervisao por observacao. A diferenca e que agora essa representacao alimenta uma arquitetura hierarquica profunda, e nao apenas uma LSTM plana.